# Missing Data Analysis

This notebook analyzes missing values in Test.csv where missing is encoded as -9999.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline


In [ ]:
df = pd.read_csv('../data/Test.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
# Identify feature columns (excluding ID)
feature_cols = [c for c in df.columns if c != 'ID']
print('Number of feature columns:', len(feature_cols))

# Assume 12 months, each with same number of features
n_months = 12
n_features_per_month = len(feature_cols) // n_months
print('Features per month:', n_features_per_month)

# Reshape to (n_rows, n_months, n_features_per_month)
data = df[feature_cols].values.reshape((df.shape[0], n_months, n_features_per_month))

missing_val = -9999

# For each month, check if any feature is missing
missing_any = np.any(data == missing_val, axis=2)  # shape (n_rows, n_months)
has_data = ~missing_any

# Months with data per row
months_with_data = has_data.sum(axis=1)
print('First 5 rows months_with_data:', months_with_data[:5])
print('Summary:')
s = pd.Series(months_with_data)
print(s.describe())
print('Mean months with data per row:', s.mean())


In [ ]:
# Per-month availability
months_avail = has_data.mean(axis=0) * 100  # percentage
print('Percentage of rows with data per month:')
for i, pct in enumerate(months_avail, start=1):
    print(f'Month {i:02d}: {pct:.2f}%')

# Plot
plt.figure(figsize=(10,5))
plt.bar(range(1,13), months_avail)
plt.xlabel('Month')
plt.ylabel('Percentage of rows with data')
plt.title('Data Availability by Month')
plt.xticks(range(1,13))
plt.ylim(0,100)
plt.show()


In [ ]:
# Number of missing months per row
missing_count = (~has_data).sum(axis=1)
print('Missing count per row statistics:')
print(pd.Series(missing_count).describe())
print()

# Distribution plot
plt.figure(figsize=(10,5))
sns.histplot(missing_count, bins=range(0,14), kde=False)
plt.xlabel('Number of missing months')
plt.ylabel('Number of rows')
plt.title('Distribution of Missing Months per Row')
plt.show()

# Longest consecutive missing months per row
def max_consecutive_missing(arr):
    max_len = cur = 0
    for v in arr:
        if v:
            cur += 1
            max_len = max(max_len, cur)
        else:
            cur = 0
    return max_len

max_consec = np.apply_along_axis(max_consecutive_missing, 1, ~has_data)
print('Maximum consecutive missing months per row:')
print(pd.Series(max_consec).describe())
print()

# Plot distribution of max consecutive missing
plt.figure(figsize=(10,5))
sns.histplot(max_consec, bins=range(0,14), kde=False)
plt.xlabel('Max consecutive missing months')
plt.ylabel('Number of rows')
plt.title('Distribution of Longest Gap of Missing Months')
plt.show()

# Check if missing pattern is monotonic (missing only at start or end)
def is_monotonic_missing(arr):
    # Find first and last missing
    idx = np.where(arr)[0]
    if len(idx) == 0:
        return True  # no missing
    return (idx[-1] - idx[0] + 1) == len(idx)  # contiguous block

monotonic = np.apply_along_axis(is_monotonic_missing, 1, ~has_data)
print('Proportion of rows with missing months forming a single contiguous block:')
print(monotonic.mean())

# Additional: count rows with missing at start only, end only, middle gaps
start_only = np.zeros(len(monotonic), dtype=bool)
end_only = np.zeros(len(monotonic), dtype=bool)
middle_gap = np.zeros(len(monotonic), dtype=bool)

for i, row in enumerate(~has_data):
    if not np.any(row):
        continue  # no missing
    first = np.where(row)[0][0]
    last = np.where(row)[0][-1]
    if first == 0 and last == np.sum(row)-1:
        # missing from start to some point
        # check if missing are at start (i.e., missing indices are 0..k)
        if np.all(np.arange(0, np.sum(row)) == np.where(row)[0]):
            start_only[i] = True
    if last == 11 and first == np.sum(row)-1 + (11 - np.sum(row)):
        # missing at end
        # missing indices are (12 - k) .. 11
        if np.all(np.arange(12 - np.sum(row), 12) == np.where(row)[0]):
            end_only[i] = True
    # else could be middle gap
    if not (start_only[i] or end_only[i]):
        middle_gap[i] = True

print('Missing at start only:', start_only.sum())
print('Missing at end only:', end_only.sum())
print('Missing in middle gaps:', middle_gap.sum())


In [ ]:
# Check for gaps in missing months considering circular continuity (Jan-Dec wrap)
def has_circular_gap(arr):
    """Return True if missing months are NOT all contiguous on a circle of 12 months."""
    idx = np.where(arr)[0]
    if len(idx) <= 1:
        return False  # 0 or 1 missing month -> no gap
    idx_sorted = np.sort(idx)
    diffs = np.diff(idx_sorted)
    circular_diff = (idx_sorted[0] + 12) - idx_sorted[-1]
    gaps = np.concatenate([diffs, [circular_diff]])
    num_gaps = np.sum(gaps > 1)
    return num_gaps > 1  # more than one gap means missing months are split into multiple arcs

circular_gap = np.apply_along_axis(has_circular_gap, 1, ~has_data)
print('Proportion of rows with missing months separated by gaps (considering Jan-Dec wrap):')
print(circular_gap.mean())


In [ ]:
# -----------------------------------------------------------
# Month availability analysis
# -----------------------------------------------------------

# Month has data if ANY of the 12 features is present
has_any_data = np.any(data != -9999, axis=2)      # (n_samples, 12)

print("Shape:", has_any_data.shape)
print("dtype:", has_any_data.dtype)

# Convert bool -> int for arithmetic
A = has_any_data.astype(np.int32)

n_samples = A.shape[0]

# -----------------------------------------------------------
# Marginal probability P(i)
# -----------------------------------------------------------

P = A.mean(axis=0)

print("\nProbability each month has data:")
for m, p in enumerate(P, start=1):
    print(f"Month {m:2d}: {p:.3f}")

# -----------------------------------------------------------
# Joint counts
# -----------------------------------------------------------

joint_counts = A.T @ A

print("\nJoint counts:")
print(joint_counts)

# -----------------------------------------------------------
# Joint probability P(i AND j)
# -----------------------------------------------------------

joint_prob = joint_counts / n_samples

print("\nJoint probability:")
print(np.round(joint_prob, 3))

# -----------------------------------------------------------
# Verify diagonal
# -----------------------------------------------------------

print("\nDiagonal of joint probability:")
print(np.round(np.diag(joint_prob), 3))

print("\nMarginal probability:")
print(np.round(P, 3))

print("\nCheck diagonal equals marginal:")
print(np.allclose(np.diag(joint_prob), P))

# -----------------------------------------------------------
# Conditional probability P(j | i)
# -----------------------------------------------------------

conditional = joint_counts / joint_counts.diagonal()[:, None]

print("\nConditional probability P(j | i):")
print(np.round(conditional, 3))

# -----------------------------------------------------------
# Plot joint probability
# -----------------------------------------------------------

plt.figure(figsize=(8,6))

sns.heatmap(
    joint_prob,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    square=True,
    xticklabels=range(1,13),
    yticklabels=range(1,13),
    vmin=0,
    vmax=joint_prob.max()
)

plt.xlabel("Month j")
plt.ylabel("Month i")
plt.title("Joint Probability P(month i AND month j)")
plt.tight_layout()
plt.show()

# -----------------------------------------------------------
# Plot conditional probability
# -----------------------------------------------------------

plt.figure(figsize=(8,6))

sns.heatmap(
    conditional,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    square=True,
    xticklabels=range(1,13),
    yticklabels=range(1,13),
    vmin=0,
    vmax=1
)

plt.xlabel("Month j")
plt.ylabel("Month i")
plt.title("Conditional Probability P(month j | month i)")
plt.tight_layout()
plt.show()